# Milestone 4: Complete Optimization Model

**Fresh Product Assortment and Stocking Allocation Across Three Dingdong Fulfillment Stores**  
**Group 21L · November 4, 2026**  
Jiayue Zhang · Yue Peng · Andrew Bian · Langzhuo Liu · Hang Seo Cho

The model selects a weekly assortment and daily new-stock deliveries for three Dingdong
fulfillment stores over June 26–July 2, 2024. Decisions are made at the June 25 cutoff and are
shared across three demand scenarios. The objective balances unmet demand, expiration, overnight
carryover, and terminal inventory subject to store inventory capacity and shared regional supply.

The inputs comprise 609 point forecasts, 1,827 scenario-demand records, and 7,830 historical
recovered-demand observations. Scenario quantities and probabilities are read from the supplied
CSV files at their recorded precision. Quantities are dataset-normalized; the objective measures
relative operational loss rather than monetary cost.


## 1. Computational environment and parameter settings

The implementation uses Python, the Gurobi Optimizer, pandas, and NumPy. Dependency requirements
are provided in `requirements.txt`. Gurobi authentication uses the local license configuration;
credentials are not included in the project files.

The base case uses an assortment cap of 20 products per store, store capacities of 51.25, 46.93,
and 46.41, and daily regional supply of 144.43 normalized units. Loss weights are
α = 5, β = 1, h = 0.2, and τ = 0.5. The solver time limit is 300 seconds, with a requested relative
MIP gap of 0.001. The achieved status and gap are reported with the solution.


In [1]:
"""Group 21L: optimization model and solution verification."""
from __future__ import annotations
import copy
import hashlib
import json
import math
from itertools import product
from pathlib import Path
import gurobipy as gp
from gurobipy import GRB
import numpy as np
import pandas as pd

STATES = ("O", "V", "A", "Sold", "Snew", "L", "W", "U")




In [2]:
PROJECT_DIR=Path.cwd()
DATA_DIR=PROJECT_DIR/"data"
OUTPUT_ROOT=PROJECT_DIR/"results"
OUTPUT_DIR=OUTPUT_ROOT/"base_case"
STORES=[18,145,235]
PRODUCTS=[21,70,79,94,117,118,136,140,207,213,215,247,249,296,366,422,496,500,
          554,556,633,634,636,638,717,738,775,797,843]
DATES=pd.date_range("2024-06-26",periods=7).strftime("%Y-%m-%d").tolist()
BASE_SETTINGS={"K":20,"capacity_by_store":{18:51.25,145:46.93,235:46.41},
               "regional_supply":144.43,"capacity_scale":1.0,"supply_scale":1.0,
               "alpha":5.0,"beta":1.0,"h":0.2,"tau":0.5,
               "probabilities":None}  # None reads the supplied CSV weights.
SOLVER_SETTINGS={"TimeLimit":300,"MIPGap":0.001,"Seed":21,"Threads":4}
CASE_OVERRIDES={}
print("Gurobi version:",".".join(map(str,gp.gurobi.version())))


Gurobi version: 13.0.3


## 2. Input validation and forecast consistency

Input validation requires the complete product of 3 stores, 29 products, 7 planning days, and
3 scenarios. Keys must be unique, dates must agree with the day index, and all demand values
must be finite and nonnegative. Each scenario has a consistent probability; probabilities sum
to one. The historical sample contains 90 complete days through June 25, and recovered demand
cannot be lower than observed sales.

The point forecast is checked against the specified recent seven-day demand level multiplied
by the store weekday index estimated over the preceding 28 days. The historical and point-forecast
tables support input validation; scenario demand enters the optimization model. SHA-256 hashes
identify the input files used in the reported computation.


In [3]:
def load_and_audit_inputs(data_dir, stores, products, dates):
    files = {"forecast": Path(data_dir) / "point_forecast_609.csv",
             "scenarios": Path(data_dir) / "scenarios_1827.csv",
             "history": Path(data_dir) / "recovered_demand.csv"}
    frames = {key: pd.read_csv(path) for key, path in files.items()}
    f, sc, hist = (frames[k] for k in ("forecast", "scenarios", "history"))
    required = {"forecast": {"store_id", "product_id", "t", "dt", "fc"},
                "scenarios": {"store_id", "product_id", "t", "dt", "scenario", "weight", "demand"},
                "history": {"store_id", "product_id", "dt", "sale_amount", "demand"}}
    for key, frame in frames.items():
        if not required[key].issubset(frame.columns) or frame[list(required[key])].isna().any().any():
            raise ValueError(f"Missing columns/values in {files[key].name}")
        for col in ("store_id", "product_id"):
            v = pd.to_numeric(frame[col], errors="raise")
            if not np.isfinite(v).all() or not (v == np.floor(v)).all():
                raise ValueError(f"Noninteger ID in {files[key].name}")
            frame[col] = v.astype(int)
        frame["dt"] = pd.to_datetime(frame["dt"], errors="raise").dt.strftime("%Y-%m-%d")
    sc["scenario"] = sc.scenario.astype(str).str.strip().str.lower()
    omega = ["low", "typical", "high"]
    expected3 = set(product(stores, products, dates))
    expected4 = set(product(stores, products, dates, omega))
    for frame, columns, expected in [(f, ["store_id", "product_id", "dt"], expected3),
                                     (sc, ["store_id", "product_id", "dt", "scenario"], expected4)]:
        if frame.duplicated(columns).any() or set(frame[columns].itertuples(index=False, name=None)) != expected:
            raise ValueError("Duplicate, missing, or extra forecast/scenario keys")
        expected_t = frame.dt.map({day: i for i, day in enumerate(dates, 1)})
        if not (pd.to_numeric(frame.t, errors="raise") == expected_t).all():
            raise ValueError("t does not match dt")
    for frame, columns in [(f, ["fc"]), (sc, ["demand", "weight"]),
                           (hist, ["sale_amount", "demand"])]:
        for col in columns:
            frame[col] = pd.to_numeric(frame[col], errors="raise")
            if not np.isfinite(frame[col]).all() or (frame[col] < 0).any():
                raise ValueError(f"Invalid values in {col}")
    hist_dates = pd.date_range("2024-03-28", "2024-06-25").strftime("%Y-%m-%d").tolist()
    hist_keys = ["store_id", "product_id", "dt"]
    if hist.duplicated(hist_keys).any() or set(hist[hist_keys].itertuples(index=False, name=None)) != set(product(stores, products, hist_dates)):
        raise ValueError("Historical recovered-demand coverage differs from the 90-day sample")
    if (hist.demand < hist.sale_amount - 1e-9).any():
        raise ValueError("Recovered demand is below observed sales")
    pi = {}
    for w in omega:
        weights = sc.loc[sc.scenario == w, "weight"].unique()
        if len(weights) != 1:
            raise ValueError("Each scenario must have one consistent weight")
        pi[w] = float(weights[0])
    if not math.isclose(sum(pi.values()), 1, abs_tol=1e-9):
        raise ValueError("Scenario probabilities do not sum to one")
    # File values remain authoritative; this checks the agreed common-multiplier structure.
    merged = sc.merge(f[["store_id", "product_id", "dt", "fc"]],
                      on=["store_id", "product_id", "dt"], validate="many_to_one")
    config_rows = []
    for w in omega:
        sub = merged[merged.scenario == w]
        ratios = sub.loc[sub.fc > 0, "demand"] / sub.loc[sub.fc > 0, "fc"]
        kappa = float(ratios.mean()) if len(ratios) else np.nan
        multiplier_error = float((sub.demand - sub.fc * kappa).abs().max())
        if not math.isfinite(kappa) or multiplier_error > 1e-8:
            raise ValueError("Scenario file does not follow a common forecast multiplier")
        config_rows.append(dict(scenario=w, probability=pi[w], multiplier=kappa,
                                total_demand=float(sub.demand.sum()),
                                max_multiplier_check_error=multiplier_error))
    # Independently check that fc follows the submitted seven-day level × store weekday index.
    recent = hist[hist.dt.between("2024-06-19", "2024-06-25")].groupby(["store_id", "product_id"]).demand.mean()
    totals = hist[hist.dt.between("2024-05-29", "2024-06-25")].groupby(["store_id", "dt"], as_index=False).demand.sum()
    totals["weekday"] = pd.to_datetime(totals.dt).dt.dayofweek
    store_mean = totals.groupby("store_id").demand.mean()
    weekday_mean = totals.groupby(["store_id", "weekday"]).demand.mean()
    implied = [recent.loc[(r.store_id, r.product_id)] *
               weekday_mean.loc[(r.store_id, pd.Timestamp(r.dt).dayofweek)] / store_mean.loc[r.store_id]
               for r in f.itertuples()]
    forecast_method_difference = float(np.max(np.abs(np.asarray(implied) - f.fc.to_numpy())))
    audit = pd.DataFrame([{"file": path.name, "rows": len(frames[key]),
                           "sha256": hashlib.sha256(path.read_bytes()).hexdigest()}
                          for key, path in files.items()])
    d = {(int(r.store_id), int(r.product_id), int(r.t), r.scenario): float(r.demand)
         for r in sc.itertuples()}
    return dict(S=list(stores), P=list(products), T=list(range(1, 8)), Omega=omega,
                dates=dict(enumerate(dates, 1)), d=d, pi=pi, scenario_configuration=pd.DataFrame(config_rows),
                input_audit=audit, forecast_method_max_difference=forecast_method_difference,
                forecast=f, scenarios=sc, history=hist)


def parameters(data, settings, overrides=None):
    cfg = copy.deepcopy(settings)
    overrides = {} if overrides is None else dict(overrides)
    if set(overrides) - set(cfg):
        raise ValueError("Unknown parameter setting")
    cfg.update(overrides)
    S, P, T = (data[k] for k in ("S", "P", "T"))
    K = {s: int(cfg["K"]) for s in S}
    if cfg["K"] != int(cfg["K"]) or not 0 <= cfg["K"] <= len(P):
        raise ValueError("Invalid assortment cap")
    C = {(s, t): float(cfg["capacity_by_store"][s]) * cfg["capacity_scale"] for s in S for t in T}
    R = {t: float(cfg["regional_supply"]) * cfg["supply_scale"] for t in T}
    if any(not math.isfinite(v) or v <= 0 for v in [*C.values(), *R.values()]):
        raise ValueError("Invalid resource limits")
    if any(not math.isfinite(cfg[k]) or cfg[k] < 0 for k in ("alpha", "beta", "h", "tau")):
        raise ValueError("Invalid penalty weight")
    pi = cfg["probabilities"] if cfg["probabilities"] is not None else data["pi"]
    if set(pi) != set(data["Omega"]) or any(not math.isfinite(v) or v < 0 for v in pi.values()) or not math.isclose(sum(pi.values()), 1, abs_tol=1e-9):
        raise ValueError("Invalid scenario weights")
    Q = {(s, p, t): min(C[s, t], R[t]) for s, p, t in product(S, P, T)}
    B = {k: max(C[k[0], k[2]], value) for k, value in data["d"].items()}
    return dict(C=C, R=R, K=K, Q=Q, B=B, pi=pi, cfg=cfg)




In [4]:
data=load_and_audit_inputs(DATA_DIR,STORES,PRODUCTS,DATES)
par=parameters(data,BASE_SETTINGS,CASE_OVERRIDES)
display(data["input_audit"])
display(data["scenario_configuration"])
print("Point forecast total:",data["forecast"].fc.sum())
print("Maximum difference from the submitted forecast rule:",data["forecast_method_max_difference"])
if "fallback" in data["forecast"]:
    print("Forecast fallback values:",data["forecast"].fallback.value_counts().to_dict())


,file,rows,sha256
0,point_forecast_609.csv,609,e741bb2eea4a3e48c8d0eb103d6d6796d95b4b985eee5f...
1,scenarios_1827.csv,1827,bb5b4655f8cf81cfa22782cf02a849d20c69aa11b289fc...
2,recovered_demand.csv,7830,dcb8bf9f1c4e2dac2354fca66552f00cf0c32e3f82df88...


,scenario,probability,multiplier,total_demand,max_multiplier_check_error
0,low,0.3,0.942545,1069.651168,3.552714e-15
1,typical,0.4,1.045028,1185.953921,3.552714e-15
2,high,0.3,1.175382,1333.886357,3.552714e-15


Point forecast total: 1134.8536209536305
Maximum difference from the submitted forecast rule: 1.7763568394002505e-15
Forecast fallback values: {False: 609}


## 3. Complete model and assumptions

Sets are stores $S$, the fixed 29-product candidate set $P$, days $T=\{1,\ldots,7\}$, and
joint demand scenarios $\Omega$. $z_{sp}\in\{0,1\}$ allows a product in a store's weekly assortment,
and $q_{spt}\ge0$ is daily new delivery. Neither decision has a scenario index. Scenario outcomes
are old stock $O$, pre-sale inventory $V$, total sales $A$, old/new sales $Sold,Snew$, remaining
new stock $L$, expired old stock $W$, and unmet demand $U$.

$$\min J=\sum_\omega\pi_\omega\left[\sum_{s,p,t}(\alpha U_{spt\omega}+\beta W_{spt\omega})
+h\sum_{s,p,t=1}^{6}L_{spt\omega}+\tau\sum_{s,p}L_{sp7\omega}\right].$$

Constraint explanations correspond to the numbered code comments below:

- **(5)** Each store permits at most $K_s=20$ products; only permitted products receive stock.
- **(6)** Initial old stock is zero. Remaining new stock becomes next day's old stock.
- **(7)** Inventory before sales equals old stock plus today's new delivery.
- **(8)** Total pre-sale inventory fits store capacity in **every** scenario.
- **(9)** Three stores share a single daily limit on **new** deliveries.
- **(10)** Exact linearization enforces $A=\min(V,d)$; sales cannot be chosen below this minimum.
- **(11)** Exact linearization enforces $Sold=\min(O,d)$, so old stock sells first.
- **(12)** Residual sales use new stock; remaining new stock carries over once.
- **(13)** Unsold old stock expires; unserved demand, including unselected products, remains unmet.
- **(14)** Selection/branch variables are binary; stock and flow variables are continuous and nonnegative.

Products are assumed sellable on arrival day and the next day. There is one delivery before daily
sales, no substitution, no inter-store transfers, and no backordering. Category codes impose no
coverage requirement. Day-7 new stock receives only the terminal charge $\tau$, not an expiration
penalty or another $h$ charge. $z=1$ permits, but does not force, positive deliveries.

Capacity and supply are historical-sales-calibrated proxies, not measured physical limits.
$\alpha=5,\beta=1,h=0.2,\tau=0.5$ are preference assumptions. Derived bounds are
$Q_{spt}=\min(C_{st},R_t)$ and $B_{spt\omega}=\max(C_{st},d_{spt\omega})$.


In [5]:
def build_model(data, par, env):
    S, P, T, Omega = (data[k] for k in ("S", "P", "T", "Omega"))
    k3, k4 = list(product(S, P, T)), list(product(S, P, T, Omega))
    d, C, Q, B, cfg = data["d"], par["C"], par["Q"], par["B"], par["cfg"]
    m = gp.Model("Group21L_Milestone4", env=env)
    v = {"z": m.addVars(S, P, vtype=GRB.BINARY, name="z"),
         "q": m.addVars(k3, lb=0, ub=Q, name="q")}
    for name in STATES:
        ub = {k: d[k] if name == "U" else min(C[k[0], k[2]], d[k])
              if name in ("A", "Sold") else C[k[0], k[2]] for k in k4}
        v[name] = m.addVars(k4, lb=0, ub=ub, name=name)
    v["bA"] = m.addVars(k4, vtype=GRB.BINARY, name="bA")
    v["bO"] = m.addVars(k4, vtype=GRB.BINARY, name="bO")
    z, q = v["z"], v["q"]
    O, V, A, Sold, Snew, L, W, U = (v[n] for n in STATES)
    # (5) Weekly assortment and delivery eligibility.
    m.addConstrs((gp.quicksum(z[s, p] for p in P) <= par["K"][s] for s in S), "assortment")
    m.addConstrs((q[k] <= Q[k] * z[k[0], k[1]] for k in k3), "delivery_selection")
    # (6) Initial old stock = 0; new stock carries for one night only.
    for s, p, t, w in k4:
        m.addConstr(O[s, p, t, w] == (0 if t == T[0] else L[s, p, t-1, w]),
                    name=f"old_stock[{s},{p},{t},{w}]")
    # (7) Inventory before sales.
    m.addConstrs((V[k] == O[k] + q[k[:3]] for k in k4), "inventory_balance")
    # (8) Capacity includes old stock and new delivery under every scenario.
    m.addConstrs((gp.quicksum(V[s, p, t, w] for p in P) <= C[s, t]
                  for s in S for t in T for w in Omega), "store_capacity")
    # (9) Shared regional limit on new deliveries.
    m.addConstrs((gp.quicksum(q[s, p, t] for s in S for p in P) <= par["R"][t] for t in T), "regional_supply")
    # (10) Exact total sales A = min(V, d).
    m.addConstrs((A[k] <= V[k] for k in k4), "sales_inventory")
    m.addConstrs((A[k] <= d[k] for k in k4), "sales_demand")
    m.addConstrs((A[k] >= V[k] - B[k] * v["bA"][k] for k in k4), "sales_branch1")
    m.addConstrs((A[k] >= d[k] - B[k] * (1-v["bA"][k]) for k in k4), "sales_branch2")
    # (11) Exact old-stock sales Sold = min(O, d), enforcing FIFO.
    m.addConstrs((Sold[k] <= O[k] for k in k4), "old_sales_inventory")
    m.addConstrs((Sold[k] <= d[k] for k in k4), "old_sales_demand")
    m.addConstrs((Sold[k] >= O[k] - B[k] * v["bO"][k] for k in k4), "old_sales_branch1")
    m.addConstrs((Sold[k] >= d[k] - B[k] * (1-v["bO"][k]) for k in k4), "old_sales_branch2")
    # (12) New-stock sales and carryover.
    m.addConstrs((Snew[k] == A[k]-Sold[k] for k in k4), "new_sales")
    m.addConstrs((L[k] == q[k[:3]]-Snew[k] for k in k4), "new_left")
    # (13) Old-stock expiration and unmet demand, including unselected products.
    m.addConstrs((W[k] == O[k]-Sold[k] for k in k4), "expired_old")
    m.addConstrs((U[k] == d[k]-A[k] for k in k4), "unmet")
    # (4) Day-7 remaining new stock is charged tau, not h or expired waste.
    objective = gp.quicksum(par["pi"][w] * (
        gp.quicksum(cfg["alpha"] * U[s,p,t,w] + cfg["beta"] * W[s,p,t,w] for s,p,t in k3)
        + cfg["h"] * gp.quicksum(L[s,p,t,w] for s in S for p in P for t in T[:-1])
        + cfg["tau"] * gp.quicksum(L[s,p,T[-1],w] for s in S for p in P)) for w in Omega)
    m.setObjective(objective, GRB.MINIMIZE)
    # Complete feasible zero-delivery MIP start; it adds no constraint or cost.
    for variables in v.values():
        for x in variables.values():
            x.Start = 0
    for k in k4:
        U[k].Start = d[k]
    m.update()
    return m, v




## 4. Solution verification and exported results

An independent inventory calculation reconstructs each demand scenario from the selected
delivery plan. Simulated sales, FIFO issuing, expiration, and carryover must agree with the
optimization variables. Additional checks cover resource limits, selection links, variable
domains, and reconciliation of objective components. A failed check prevents result export.

The computation records the incumbent objective, best bound, relative MIP gap, runtime, and
termination status. A feasible incumbent at the time limit is distinguished from a solution
meeting the requested optimality tolerance.


In [6]:
def extract_and_audit(data, par, m, v, tolerance=1e-5):
    S, P, T, Omega = (data[k] for k in ("S", "P", "T", "Omega"))
    cfg = par["cfg"]
    q = {k: float(x.X) for k, x in v["q"].items()}
    z = {k: int(round(x.X)) for k, x in v["z"].items()}
    assortment = pd.DataFrame([dict(store_id=s, product_id=p, selected=z[s,p],
                                    weekly_delivery=sum(q[s,p,t] for t in T)) for s,p in product(S,P)])
    deliveries = pd.DataFrame([dict(store_id=s, product_id=p, dt=data["dates"][t], delivery=q[s,p,t])
                               for s,p,t in product(S,P,T)])
    rows, state_error = [], 0.0
    for s,p,w in product(S,P,Omega):
        old = 0.0
        for t in T:
            k=(s,p,t,w)
            D, new = data["d"][k], q[s,p,t]
            avail = old + new
            sold_old, sales = min(old,D), min(avail,D)
            sold_new = sales-sold_old
            left = new-sold_new
            simulated = dict(O=old,V=avail,A=sales,Sold=sold_old,Snew=sold_new,L=left,W=old-sold_old,U=D-sales)
            actual = {n: float(v[n][k].X) for n in STATES}
            state_error=max(state_error,max(abs(actual[n]-simulated[n]) for n in STATES))
            rows.append(dict(store_id=s,product_id=p,dt=data["dates"][t],day=t,scenario=w,
                             demand=D,delivery=new,selected=z[s,p],**actual))
            old=left
    outcomes=pd.DataFrame(rows)
    capacity=outcomes.groupby(["store_id","dt","day","scenario"],as_index=False).V.sum()
    capacity["capacity"]=[par["C"][s,t] for s,t in zip(capacity.store_id,capacity.day)]
    capacity["slack"]=capacity.capacity-capacity.V
    capacity["utilization"]=capacity.V/capacity.capacity
    supply=deliveries.groupby("dt",as_index=False).delivery.sum()
    supply["limit"]=[par["R"][t] for t in T]
    supply["slack"]=supply.limit-supply.delivery
    checks=dict(max_constraint_violation=float(m.ConstrVio),max_bound_violation=float(m.BoundVio),
                max_integrality_violation=float(m.IntVio),max_independent_inventory_error=state_error,
                max_capacity_violation=max(0.0,-float(capacity.slack.min())),
                max_supply_violation=max(0.0,-float(supply.slack.min())),
                max_assortment_violation=max(max(0,sum(z[s,p] for p in P)-par["K"][s]) for s in S),
                max_delivery_selection_violation=max(max(0,q[s,p,t]-par["Q"][s,p,t]*z[s,p]) for s,p,t in product(S,P,T)))
    if max(checks.values())>tolerance:
        raise RuntimeError(f"Solution audit failed: {checks}")
    scenario_rows,store_rows,component_rows=[],[],[]
    for w in Omega:
        sub=outcomes[outcomes.scenario==w]
        D,sales,unmet,expired=map(float,(sub.demand.sum(),sub.A.sum(),sub.U.sum(),sub.W.sum()))
        carry=float(sub.loc[sub.day<T[-1],"L"].sum())
        terminal=float(sub.loc[sub.day==T[-1],"L"].sum())
        omitted=float(sub.loc[sub.selected==0,"demand"].sum())
        comp=dict(unmet_demand=cfg["alpha"]*unmet,expired_stock=cfg["beta"]*expired,
                  overnight_carryover=cfg["h"]*carry,terminal_new_stock=cfg["tau"]*terminal)
        for label,value in comp.items():
            component_rows.append(dict(scenario=w,component=label,loss=value,probability=par["pi"][w],weighted_loss=value*par["pi"][w]))
        scenario_rows.append(dict(scenario=w,probability=par["pi"][w],demand=D,sales=sales,fill_rate=sales/D if D else np.nan,
                                  unmet_demand=unmet,unselected_product_demand=omitted,selected_product_unmet=unmet-omitted,
                                  expired_stock=expired,overnight_carryover=carry,day7_new_stock=terminal,loss=sum(comp.values())))
        for s in S:
            ss=sub[sub.store_id==s]
            total,served=float(ss.demand.sum()),float(ss.A.sum())
            store_rows.append(dict(store_id=s,scenario=w,demand=total,sales=served,fill_rate=served/total if total else np.nan,
                                   unmet_demand=float(ss.U.sum()),expired_stock=float(ss.W.sum()),
                                   carryover=float(ss.loc[ss.day<T[-1],"L"].sum()),day7_new_stock=float(ss.loc[ss.day==T[-1],"L"].sum())))
    components=pd.DataFrame(component_rows)
    checks["objective_reconciliation_error"]=abs(float(components.weighted_loss.sum())-float(m.ObjVal))
    if checks["objective_reconciliation_error"]>tolerance*max(1,abs(m.ObjVal)):
        raise RuntimeError("Objective components do not reconcile")
    return dict(assortment=assortment,deliveries=deliveries,scenario_outcomes=outcomes,
                scenario_summary=pd.DataFrame(scenario_rows),store_scenario_summary=pd.DataFrame(store_rows),
                objective_components=components,store_daily_capacity=capacity,regional_daily_supply=supply,checks=checks)


def solve_case(data, settings, solver_settings, output_dir, overrides=None):
    output_dir=Path(output_dir)
    output_dir.mkdir(parents=True,exist_ok=True)
    par=parameters(data,settings,overrides)
    with gp.Env(empty=True) as env:
        env.setParam("OutputFlag",0)
        env.start()
        m,v=build_model(data,par,env)
        try:
            m.Params.LogToConsole=0
            m.Params.LogFile=str(output_dir/"gurobi.log")
            m.Params.OutputFlag=1
            for name,value in solver_settings.items():
                m.setParam(name,value)
            m.write(str(output_dir/"model.lp"))
            m.optimize()
            if m.SolCount==0:
                if m.Status==GRB.INFEASIBLE:
                    m.computeIIS()
                    m.write(str(output_dir/"infeasible.ilp"))
                raise RuntimeError(f"Gurobi status {m.Status}; no candidate solution")
            result=extract_and_audit(data,par,m,v)
            summary=dict(input_source="forecast_team_csv",
                         status="OPTIMAL_WITHIN_CONFIGURED_GAP" if m.Status==GRB.OPTIMAL else "FEASIBLE_INCUMBENT_NOT_PROVEN_OPTIMAL",
                         gurobi_status_code=int(m.Status),objective_weighted_loss=float(m.ObjVal),best_bound=float(m.ObjBound),
                         mip_gap=float(m.MIPGap),configured_mip_gap=float(m.Params.MIPGap),runtime_seconds=float(m.Runtime),
                         variables=int(m.NumVars),binary_variables=int(m.NumBinVars),constraints=int(m.NumConstrs),
                         gurobi_version=list(gp.gurobi.version()),checks=result["checks"],
                         selected_products_by_store={str(s):int(result["assortment"].loc[result["assortment"].store_id==s,"selected"].sum()) for s in data["S"]})
            for name,table in result.items():
                if isinstance(table,pd.DataFrame): table.to_csv(output_dir/f"{name}.csv",index=False)
            data["input_audit"].to_csv(output_dir/"input_audit.csv",index=False)
            data["scenario_configuration"].to_csv(output_dir/"scenario_configuration.csv",index=False)
            (output_dir/"solver_summary.json").write_text(json.dumps(summary,indent=2)+"\n")
            (output_dir/"run_parameters.json").write_text(json.dumps(dict(model=par["cfg"],scenario_weights=par["pi"],solver=solver_settings),indent=2)+"\n")
            m.write(str(output_dir/"solution.sol"))
        finally:
            m.dispose()
    result["solver_summary"]=summary
    return result


## 5. Base-case solution

The optimization uses the validated scenario-demand table and the stated base-case parameters.
The complete zero-delivery MIP start is feasible and introduces no additional constraint or
objective term. Exported artifacts include the algebraic model, incumbent solution, solver log,
active parameters, input hashes, and decision and outcome tables.


In [7]:
print("Base-case solver time limit:",SOLVER_SETTINGS["TimeLimit"],"seconds.")
results=solve_case(data,BASE_SETTINGS,SOLVER_SETTINGS,OUTPUT_DIR,CASE_OVERRIDES)
summary=results["solver_summary"]
display(pd.Series({k:summary[k] for k in ["status","objective_weighted_loss","best_bound","mip_gap",
                                        "runtime_seconds","variables","constraints"]},name="Gurobi result"))
display(results["scenario_summary"].round(6))


Base-case solver time limit: 300 seconds.


status                     FEASIBLE_INCUMBENT_NOT_PROVEN_OPTIMAL
objective_weighted_loss                              1229.305778
best_bound                                           1212.694853
mip_gap                                                 0.013512
runtime_seconds                                       300.162215
variables                                                  18966
constraints                                                26260
Name: Gurobi result, dtype: object

,scenario,probability,demand,sales,fill_rate,unmet_demand,unselected_product_demand,selected_product_unmet,expired_stock,overnight_carryover,day7_new_stock,loss
0,low,0.3,1069.651168,939.617742,0.878434,130.033426,119.538323,10.495103,9.326935,39.446378,9.168995,671.967838
1,typical,0.4,1185.953921,956.354893,0.806401,229.599028,132.535678,97.063350,1.758778,31.233080,0.000000,1156.000534
2,high,0.3,1333.886357,958.113671,0.718287,375.772686,149.067792,226.704894,0.000000,27.603069,0.000000,1884.384045


## 6. Selection, delivery, and resource use

The assortment and delivery plan is shared across scenarios. Each scenario's sales, shortages,
expiration, and carryover result from that same plan. The resource tables distinguish unused
new-supply allowance from store inventory capacity occupied by carryover.


In [8]:
display(results["assortment"].groupby("store_id").agg(
    selected_products=("selected","sum"),weekly_delivery=("weekly_delivery","sum")).round(6))
display(results["deliveries"].pivot_table(index="store_id",columns="dt",values="delivery",aggfunc="sum").round(5))
display(results["regional_daily_supply"].round(6))
display(results["store_scenario_summary"].round(6))
display(results["store_daily_capacity"].groupby(["store_id","scenario"]).agg(
    maximum_utilization=("utilization","max"),minimum_slack=("slack","min")).round(6))


,selected_products,weekly_delivery
store_id,,
18,20,343.311772
145,20,316.834197
235,20,297.967701


dt,2024-06-26,2024-06-27,2024-06-28,2024-06-29,2024-06-30,2024-07-01,2024-07-02
store_id,,,,,,,
18,51.14485,47.02521,47.61853,48.52614,49.39814,48.34889,51.250
145,46.74907,44.77696,42.03441,44.97446,46.93000,44.43931,46.930
235,46.34931,38.87632,34.30586,45.10741,46.41000,42.03081,44.888


,dt,delivery,limit,slack
0,2024-06-26,144.243230,144.43,0.186770
1,2024-06-27,130.678492,144.43,13.751508
2,2024-06-28,123.958799,144.43,20.471201
3,2024-06-29,138.608002,144.43,5.821998
4,2024-06-30,142.738141,144.43,1.691859
5,2024-07-01,134.819012,144.43,9.610988
6,2024-07-02,143.067995,144.43,1.362005


,store_id,scenario,demand,sales,fill_rate,unmet_demand,expired_stock,carryover,day7_new_stock
0,18,low,384.865211,340.523774,0.884787,44.341437,2.787998,12.175102,0.000000
1,145,low,356.327785,311.697643,0.874750,44.630141,1.891588,7.908015,3.244966
2,235,low,328.458172,287.396324,0.874986,41.061848,4.647348,19.363262,5.924029
3,18,typical,426.711455,343.093753,0.804042,83.617702,0.218019,9.598834,0.000000
4,145,typical,395.071165,316.592204,0.801355,78.478962,0.241994,6.411432,0.000000
5,235,typical,364.171300,296.668936,0.814641,67.502364,1.298765,15.222814,0.000000
6,18,high,479.938199,343.311772,0.715325,136.626426,0.000000,8.445260,0.000000
7,145,high,444.351191,316.834197,0.713027,127.516994,0.000000,5.649277,0.000000
8,235,high,409.596967,297.967701,0.727466,111.629266,0.000000,13.508531,0.000000


maximum_utilization  minimum_slack
store_id scenario                                    
18       high                      1.0            0.0
         low                       1.0           -0.0
         typical                   1.0            0.0
145      high                      1.0           -0.0
         low                       1.0           -0.0
         typical                   1.0           -0.0
235      high                      1.0            0.0
         low                       1.0           -0.0
         typical                   1.0           -0.0

In [9]:
display(pd.Series(results["checks"],name="verification"))
display(results["objective_components"].groupby("component").weighted_loss.sum().to_frame().round(6))
regional_weekly_limit=sum(par["R"].values())
diagnostics=results["scenario_summary"][["scenario","demand","fill_rate","unmet_demand",
                                      "unselected_product_demand","selected_product_unmet"]].copy()
diagnostics["minimum_shortage_from_supply_alone"]=(diagnostics.demand-regional_weekly_limit).clip(lower=0)
display(diagnostics.round(6))
print("All quantities are normalized; these are scenario results, not measured business outcomes.")
print("Results saved under results/base_case.")


max_constraint_violation            3.392842e-13
max_bound_violation                 0.000000e+00
max_integrality_violation           0.000000e+00
max_independent_inventory_error     3.392842e-13
max_capacity_violation              2.060574e-13
max_supply_violation                0.000000e+00
max_assortment_violation            0.000000e+00
max_delivery_selection_violation    0.000000e+00
objective_reconciliation_error      0.000000e+00
Name: verification, dtype: float64

,weighted_loss
component,
expired_stock,3.501592
overnight_carryover,6.521613
terminal_new_stock,1.375349
unmet_demand,1217.907224


,scenario,demand,fill_rate,unmet_demand,unselected_product_demand,selected_product_unmet,minimum_shortage_from_supply_alone
0,low,1069.651168,0.878434,130.033426,119.538323,10.495103,58.641168
1,typical,1185.953921,0.806401,229.599028,132.535678,97.063350,174.943921
2,high,1333.886357,0.718287,375.772686,149.067792,226.704894,322.876357


All quantities are normalized; these are scenario results, not measured business outcomes.
Results saved under results/base_case.


## 7. Interpretation and limitations

The reported computation terminated after 300 seconds with a feasible incumbent objective of
1,229.3058, a best bound of 1,212.6949, and a relative MIP gap of 1.3512%. Independent inventory
and constraint checks passed. Each store selected 20 products, and weekly deliveries totaled
958.1137 normalized units.

Unselected-product demand is included in unmet demand. In the low-demand scenario, 119.5383 of
130.0334 unmet units correspond to excluded products. Store inventory capacity binds in 12 of
21 low-scenario store-day observations. Regional new-supply limits have positive slack on all
seven days in this incumbent. These results indicate that assortment restrictions and inventory
capacity are material features of the reported plan; they do not establish the marginal value
of changing a constraint without additional optimization.

The two-day sellable life, resource proxies, and relative penalties are modeling assumptions.
Demand during historical stockouts is estimated, and the common scenario multiplier does not
represent product-specific demand shocks. Results describe modeled demand scenarios; they do
not establish realized business savings or improvement over a comparison policy.

**Data and software references.** [FreshRetailNet-50K](https://huggingface.co/datasets/Dingdong-Inc/FreshRetailNet-50K);
[Gurobi Python API](https://docs.gurobi.com/projects/optimizer/en/current/reference/python/model.html).

**AI-use disclosure.** AI tools assisted code development and documentation. Explicit input,
constraint, inventory-flow, and objective checks are included in this notebook.
